# Explainability Evaluation - Stability

## Testing the Attention based Explainability and SHAP on Stability

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Connect and Access Git Repository

import os
import sys
import json
from getpass import getpass
from pathlib import Path

GITHUB_USERNAME = "sharaffatima"
REPO_NAME = "bachelor-thesis"

token = getpass("GitHub token: ")

REPO_URL = f"https://{token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
REPO_DIR = Path("/content") / REPO_NAME


if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
sys.path.append(str(REPO_DIR))

print("Repo exists:", REPO_DIR.exists())

GitHub token: ··········
/content/bachelor-thesis
Already up to date.
/content/bachelor-thesis
Repo exists: True


In [ ]:
# Imports
import os
import numpy as np
import pandas as pd
import shap
import torch
from torch.utils.data import DataLoader, TensorDataset

# global configuration

REPO_DIR = Path("/content/bachelor-thesis")
sys.path.append(str(REPO_DIR))


from utils.dataset import PTBXLDataset
from utils.seed import set_seed, RANDOM_SEED
from explainability.anoformer_attention import AnoFormerExplainability
from explainability.tranad_attention import TranADExplainability
from models.anoformer import AnoFormerMainModel
from models.tranad import TranADMainModel
from explainability.evaluation import StabilityEvaluation
from explainability.shap_wrappers import AnoFormerAnomalyScoreWrapper, TranADAnomalyScoreWrapper
from explainability.sample_selection import load_samples_from_dataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(RANDOM_SEED)
print(f"Active device: {DEVICE}")

Active device: cuda


In [4]:
# Paths to Directories
# Load processed data and pretrained models

DATA_DIR = Path("/content/drive/MyDrive/ECG data")
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = DATA_DIR / "results"

MODEL_DIR = RESULTS_DIR / "models"
ANOFORMER_MODEL_PATH = MODEL_DIR / "anoformer" / "anoformer_40epochs_larger_capacity_2.pt"
TRANAD_MODEL_PATH = MODEL_DIR / "tranad" / "tranad_20epochs_larger_capacity.pt"

SHAP_RESULTS_DIR = RESULTS_DIR / "shap"

STABILITY_DIR = RESULTS_DIR / "evaluation" / "stability"
STABILITY_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
# Load my preprocessed training and test datasets

X_train = np.load(PROCESSED_DIR / "X_train.npy")
y_train = np.load(PROCESSED_DIR / "y_train.npy")

X_test = np.load(os.path.join(PROCESSED_DIR, "X_test.npy"))
y_test = np.load(os.path.join(PROCESSED_DIR, "y_test.npy"))

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(17418, 1000, 12) (17418,)
(2198, 1000, 12) (2198,)


In [6]:
# select only normal ecgs from Training set
normal_train_mask = y_train == 0
X_train_normal = X_train[normal_train_mask]
y_train_normal = y_train[normal_train_mask]

# Datasets
# We need the Train dataset because the SHAP background is from the training set. We also need the mean and std of the training set to normalize the test set.
train_dataset = PTBXLDataset(
    X_train_normal,
    y_train_normal,
    normalize=True
)

# We need the Test dataset because the chosen samples for evaluation are from the test set
test_dataset = PTBXLDataset(
    X_test,
    y_test,
    mean=train_dataset.mean,
    std=train_dataset.std,
    normalize=True
)

# Dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    drop_last=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False
)

In [7]:
def make_single_ecg_loader(ecg):
    if isinstance(ecg, np.ndarray):
        ecg = torch.tensor(ecg, dtype=torch.float32)

    if ecg.ndim == 2:
        ecg = ecg.unsqueeze(0)

    dummy_label = torch.zeros(ecg.shape[0], dtype=torch.long)
    dataset = TensorDataset(ecg, dummy_label)

    return DataLoader(dataset, batch_size=1, shuffle=False)

In [ ]:
# Load fixed evaluation samples

EvaluationSamplePath = (
    RESULTS_DIR
    / "evaluation"
    / "evaluation_samples.npz"
)

EvaluationSamples = load_samples_from_dataset(
    sample_path=EvaluationSamplePath,
    test_dataset=test_dataset,
    device=None,
)

print("Number of evaluation samples:", len(EvaluationSamples))

for sample in EvaluationSamples:
    print(
        sample["index"],
        sample["case_type"],
        sample["label"],
        sample["ecg"].shape,
    )

In [ ]:
# Load previously calculated Anomaly Thresholds
# these were calculated based on the evaluation set

filepath = MODEL_DIR / "chosen_thresholds.json"

with open(filepath, "r") as f:
    thresholds = json.load(f)

AnoFormerChosenThreshold = thresholds["anoformer"]
TranADChosenThreshold = thresholds["tranad"]

In [9]:
# Create the Evaluator

StabilityEvaluator = StabilityEvaluation(
    noise_ratio=0.01,
    epsilon=1e-8
)

# AnoFormer Evaluation

In [ ]:
# Create AnoFormer Model

AnoFormer = AnoFormerMainModel(
    NumFeatures=12,
    ModelDimension=64,
    NumHeads=4,
    NumEncoderLayers=2,
    NumDecoderLayers=2,
    DiscriminatorLayers=1,
    ForwardDimension=128,
    DropoutRate=0.1,
    GeneratorLearningRate=1e-4,
    DiscriminatorLearningRate=1e-4,
    MaskingRate=0.15,
    AnomalyThreshold=0.75,
    ReconstructionWeight=10.0,
)

# Load the pretrained model
AnoFormer.Load(str(ANOFORMER_MODEL_PATH))

# Create Explainer to extract Attention Values
AnoFormerExplainer = AnoFormerExplainability(AnoFormer)


Modell geladen von: /content/drive/MyDrive/ECG data/results/models/anoformer/anoformer_40epochs_larger_capacity_2.pt


In [11]:
def anoformer_score_function(ecg):
    """
    Return the AnoFormer anomaly score for one ECG
    """

    single_loader = make_single_ecg_loader(ecg)

    scores, _ = AnoFormer.DetectAnomalies(
        TestLoader=single_loader,
        Threshold=AnoFormerChosenThreshold,
        NumMaskRuns=5,
        Seed=RANDOM_SEED,
    )

    return scores[0]

### Attention Evaluation

In [ ]:
OriginalImportanceValues = []
NoisyImportanceValues = [] # List of Lists, each ECG has a list of the importance values of its noisy variants 

for sample in EvaluationSamples:
    ecg = sample["ecg"].to(AnoFormer.Device)

    original_importance = AnoFormerExplainer.get_attention_importance(
        TimeSeriesWindow=ecg,
        BlockIndex=0,
        HeadIndex=None,
    )

    noisy_importances = []

    for _ in range(5):
        noisy_ecg = StabilityEvaluator.add_gaussian_noise(ecg)

        noisy_importance = AnoFormerExplainer.get_attention_importance(
            TimeSeriesWindow=noisy_ecg,
            BlockIndex=0,
            HeadIndex=None,
        )

        noisy_importances.append(noisy_importance)

    OriginalImportanceValues.append(original_importance)
    NoisyImportanceValues.append(noisy_importances)

In [ ]:
# Evaluate

AnoFormerAttentionStabilityResults = pd.DataFrame(
    StabilityEvaluator.evaluate_samples(
        samples=EvaluationSamples,
        original_importance_values=OriginalImportanceValues,
        noisy_importance_values=NoisyImportanceValues,
        model_name="AnoFormer",
        explanation_name="Attention",
    )
)

AnoFormerAttentionStabilityResults

In [ ]:
AnoFormerAttentionStabilitySummary = (
    AnoFormerAttentionStabilityResults
    .groupby(["Model", "Explanation", "Case Type"])["Stability Score"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

AnoFormerAttentionStabilitySummary

In [ ]:
AnoFormerAttentionStabilityResults.to_csv(
    STABILITY_DIR / "anoformer_attention_stability_results.csv",
    index=False,
)

AnoFormerAttentionStabilitySummary.to_csv(
    STABILITY_DIR / "anoformer_attention_stability_summary.csv",
    index=False,
)

## SHAP Evaluation

In [12]:
# Load the background indices used for SHAP explanations

background_indices = np.load(
    SHAP_RESULTS_DIR / "medium_background_indices.npy"
)

background_ECGs = torch.as_tensor(
    X_train[background_indices],
    dtype=torch.float32,
    device=DEVICE,
)

In [ ]:
# Load the original shap values

AnoFormerSHAPValues = np.load(
    SHAP_RESULTS_DIR
    / "anoformer"
    / "anoformer_evaluation_shap_values.npy"
)

print("AnoFormer SHAP values:", AnoFormerSHAPValues.shape)

In [ ]:
# because SHAP takes very long, choose a smaller subset to evaluate

num_samples_per_case = 5         # 5 TP, 5 TN
num_noisy_variants = 2

TNPositions = [
    position
    for position, sample in enumerate(EvaluationSamples)
    if sample["case_type"] == "TN"
]

TPPositions = [
    position
    for position, sample in enumerate(EvaluationSamples)
    if sample["case_type"] == "TP"
]

SHAPStabilityPositions = (
    TNPositions[:num_samples_per_case]
    + TPPositions[:num_samples_per_case]
)

SHAPStabilitySamples = [
    EvaluationSamples[position]
    for position in SHAPStabilityPositions
]

print("Number of SHAP stability samples:", len(SHAPStabilitySamples))
print("Positions:", SHAPStabilityPositions)

In [ ]:
# Create the same Anomaly Score Wrapper for AnoFormer that was used during SHAP explanations

AnoFormerScoreWrapper = AnoFormerAnomalyScoreWrapper(
    anoformer_model=AnoFormer,
    window_size=1000,
    masking_rate=None,
    num_masks=1,
    seed=RANDOM_SEED,
).to(DEVICE)

AnoFormerScoreWrapper.eval()

# create Explainer
AnoFormerSHAPExplainer = shap.GradientExplainer(
    AnoFormerScoreWrapper,
    background_ECGs,
)

In [ ]:
AnoFormerSHAPOriginalImportanceValues = []
AnoFormerSHAPNoisyImportanceValues = []

for position, sample in zip(SHAPStabilityPositions, SHAPStabilitySamples):
    ecg = sample["ecg"].to(AnoFormer.Device)

    original_shap_values = AnoFormerSHAPValues[position]
    original_importance = np.abs(original_shap_values).sum(axis=1)

    noisy_importances = []

    for _ in range(num_noisy_variants):

        # create a noisy variant and calculate it's shap values
        noisy_ecg = StabilityEvaluator.add_gaussian_noise(ecg)
        noisy_shap_values = AnoFormerSHAPExplainer.shap_values(noisy_ecg)

        if isinstance(noisy_shap_values, list):
            noisy_shap_values = noisy_shap_values[0]

        noisy_shap_values = np.asarray(noisy_shap_values)

        if noisy_shap_values.ndim == 4:
            noisy_shap_values = noisy_shap_values[..., 0]

        noisy_shap_values = noisy_shap_values[0]

        noisy_importance = np.abs(noisy_shap_values).sum(axis=1)

        noisy_importances.append(noisy_importance)

    AnoFormerSHAPOriginalImportanceValues.append(original_importance)
    AnoFormerSHAPNoisyImportanceValues.append(noisy_importances)

    print(
        "Computed AnoFormer SHAP stability importances for index:",
        sample["index"],
        "case:",
        sample["case_type"]
    )

In [ ]:
# Evaluate

AnoFormerSHAPStabilityList = (
    StabilityEvaluator.evaluate_samples(
        samples=SHAPStabilitySamples,
        original_importance_values=AnoFormerSHAPOriginalImportanceValues,
        noisy_importance_values=AnoFormerSHAPNoisyImportanceValues,
        model_name="AnoFormer",
        explanation_name="SHAP",
    )
)

AnoFormerSHAPStabilityResults = pd.DataFrame(
    AnoFormerSHAPStabilityList
)

AnoFormerSHAPStabilityResults

In [ ]:
AnoFormerSHAPStabilitySummary = (
    AnoFormerSHAPStabilityResults
    .groupby(
        [
            "Model",
            "Explanation",
            "Case Type",
        ]
    )["Stability Score"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

AnoFormerSHAPStabilitySummary

In [ ]:
# Save

AnoFormerSHAPStabilityResults.to_csv(
    STABILITY_DIR / "anoformer_shap_stability_results.csv",
    index=False,
)

AnoFormerSHAPStabilitySummary.to_csv(
    STABILITY_DIR / "anoformer_shap_stability_summary.csv",
    index=False,
)

# TranAD Evaluation

In [ ]:
# Create TranAD Model

TranAD = TranADMainModel(
    NumFeatures=12,
    ModelDimension=64,
    NumHeads=4,
    NumEncoderLayers=2,
    NumDecoderLayers=2,
    ForwardDimension=128,
    DropoutRate=0.1,
    LearningRate=1e-4,
    FocusGrowthRate=1.05,
    InitialFocusScore=1.0,
    MaxFocusScore=5.0,
    AnomalyPercentile=95.0,
)

# Load the pretrained model
TranAD.Load(str(TRANAD_MODEL_PATH))

# Create Explainer to extract Attention Values
TranADExplainer = TranADExplainability(TranAD)


TranADModel: 268,376 Parameter | d_model=64, H=4, Enc=2, Dec×2=2
Model loaded  /content/drive/MyDrive/ECG data/results/models/tranad/tranad_20epochs_larger_capacity.pt


In [20]:
def tranad_score_function(ecg):
    """
    Return the TranAD anomaly score for one ECG
    """

    single_loader = make_single_ecg_loader(ecg)

    scores, _ = TranAD.DetectAnomalies(
        TestLoader=single_loader,
        Threshold=TranADChosenThreshold,
    )

    return scores[0]

## Attention Evaluation

In [ ]:
OriginalImportanceValues = []
NoisyImportanceValues = []

for sample in EvaluationSamples:
    ecg = sample["ecg"].to(TranAD.Device)

    original_importance = TranADExplainer.get_attention_importance(
        TimeSeriesWindow=ecg,
        LayerIndex=0,
        HeadIndex=None,
    )

    noisy_importances = []

    for _ in range(5):
        noisy_ecg = StabilityEvaluator.add_gaussian_noise(ecg)

        noisy_importance = TranADExplainer.get_attention_importance(
            TimeSeriesWindow=ecg,
            LayerIndex=0,
            HeadIndex=None,
        )

        noisy_importances.append(noisy_importance)

    OriginalImportanceValues.append(original_importance)
    NoisyImportanceValues.append(noisy_importances)

In [ ]:
# Evaluate

TranADAttentionStabilityResults = pd.DataFrame(
    StabilityEvaluator.evaluate_samples(
        samples=EvaluationSamples,
        original_importance_values=OriginalImportanceValues,
        noisy_importance_values=NoisyImportanceValues,
        model_name="TranAD",
        explanation_name="Attention",
    )
)

TranADAttentionStabilityResults

In [ ]:
TranADAttentionStabilitySummary = (
    TranADAttentionStabilityResults
    .groupby(["Model", "Explanation", "Case Type"])["Stability Score"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

TranADAttentionStabilitySummary

In [ ]:
# Save the Results
TranADAttentionStabilityResults.to_csv(
    STABILITY_DIR / "tranad_attention_stability_results.csv",
    index=False,
)

TranADAttentionStabilitySummary.to_csv(
    STABILITY_DIR / "tranad_attention_stability_summary.csv",
    index=False,
)

## SHAP Evaluation

In [ ]:
# Load the original shap values

TranADSHAPValues = np.load(
    SHAP_RESULTS_DIR
    / "tranad"
    / "tranad_evaluation_shap_values.npy"
)

print("TranAD SHAP values:", TranADSHAPValues.shape)

In [ ]:
# Create the same TranAD Anomaly Score Wrapper for TranAD that was used during SHAP explanations

TranADScoreWrapper = TranADAnomalyScoreWrapper(TranAD)

TranADScoreWrapper.eval()

# Create the same Explainer
TranADSHAPExplainer = shap.GradientExplainer(
    TranADScoreWrapper,
    background_ECGs,
)

In [ ]:
TranADSHAPOriginalImportanceValues = []
TranADSHAPNoisyImportanceValues = []

for position, sample in zip(SHAPStabilityPositions, SHAPStabilitySamples):
    ecg = sample["ecg"].to(TranAD.Device)

    original_shap_values = TranADSHAPValues[position]
    original_importance = np.abs(original_shap_values).sum(axis=1)

    noisy_importances = []

    for _ in range(num_noisy_variants):

        # create a noisy variant and calculate it's shap values
        noisy_ecg = StabilityEvaluator.add_gaussian_noise(ecg)
        noisy_shap_values = TranADSHAPExplainer.shap_values(noisy_ecg)

        if isinstance(noisy_shap_values, list):
            noisy_shap_values = noisy_shap_values[0]

        noisy_shap_values = np.asarray(noisy_shap_values)

        if noisy_shap_values.ndim == 4:
            noisy_shap_values = noisy_shap_values[..., 0]

        noisy_shap_values = noisy_shap_values[0]

        noisy_importance = np.abs(noisy_shap_values).sum(axis=1)

        noisy_importances.append(noisy_importance)

    TranADSHAPOriginalImportanceValues.append(original_importance)
    TranADSHAPNoisyImportanceValues.append(noisy_importances)

    print(
        "Computed TranAD SHAP stability importances for index:",
        sample["index"],
        "case:",
        sample["case_type"]
    )

In [ ]:
# Evaluate

TranADSHAPStabilityList = (
    StabilityEvaluator.evaluate_samples(
        samples=SHAPStabilitySamples,
        original_importance_values=TranADSHAPOriginalImportanceValues,
        noisy_importance_values=TranADSHAPNoisyImportanceValues,
        model_name="TranAD",
        explanation_name="SHAP",
    )
)

TranADSHAPStabilityResults = pd.DataFrame(
    TranADSHAPStabilityList
)

TranADSHAPStabilityResults

In [ ]:
TranADSHAPStabilitySummary = (
    TranADSHAPStabilityResults
    .groupby(
        [
            "Model",
            "Explanation",
            "Case Type",
        ]
    )["Stability Score"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

TranADSHAPStabilitySummary

In [ ]:
# Save

TranADSHAPStabilityResults.to_csv(
    STABILITY_DIR / "tranad_shap_stability_results.csv",
    index=False,
)

TranADSHAPStabilitySummary.to_csv(
    STABILITY_DIR / "tranad_shap_stability_summary.csv",
    index=False,
)

# Final Stability Evaluation Results

In [ ]:
StabilitySummary = pd.concat(
    [
        AnoFormerAttentionStabilitySummary,
        AnoFormerSHAPStabilitySummary,
        TranADAttentionStabilitySummary,
        TranADSHAPStabilitySummary,
    ],
    ignore_index=True,
)

StabilitySummary

In [ ]:
StabilitySummary.to_csv(
    STABILITY_DIR / "reconstruction_error_agreement_summary_all_methods.csv",
    index=False,
)